 Gini impurity and entropy


In [3]:
import math 

def gini_impurity(labels):
    n=len(labels)
    if n==0:
        return 0.0
    counts={}
    for label in labels:
        counts[label]=counts.get(label,0)+1
    return 1.0 - sum((c/n)**2 for c in counts.values())
        

def entropy(labels):
    n = len(labels)
    if n == 0:
        return 0.0
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return -sum((c/n) * math.log2(c/n) for c in counts.values() if c > 0)

In [4]:
# ---- Test 1: pure (only one class) ----
labels1 = ['cat', 'cat', 'cat', 'cat']
print("Test 1:", labels1)
print("  Gini   =", gini_impurity(labels1))   # 0.0
print("  Entropy=", entropy(labels1))         # 0.0

# ---- Test 2: 50/50 split ----
labels2 = ['cat', 'dog', 'cat', 'dog']
print("\nTest 2:", labels2)
print("  Gini   =", gini_impurity(labels2))   # 0.5
print("  Entropy=", entropy(labels2))         # 1.0

# ---- Test 3: 3 cats, 2 dogs ----
labels3 = ['cat', 'dog', 'cat', 'cat', 'dog']
print("\nTest 3:", labels3)
print("  Gini   =", round(gini_impurity(labels3), 4))  # 0.48
print("  Entropy=", round(entropy(labels3), 4))        # 0.971

# ---- Test 4: empty list ----
print("\nTest 4: []")
print("  Gini   =", gini_impurity([]))         # 0.0
print("  Entropy=", entropy([]))               # 0.0

# ---- Test 5: counting trace ----
print("\n--- counting trace ---")
labels = ['a', 'b', 'a', 'c', 'a', 'b']
counts = {}
for label in labels:
    counts[label] = counts.get(label, 0) + 1
    print(f"  after seeing {label!r}: {counts}")

Test 1: ['cat', 'cat', 'cat', 'cat']
  Gini   = 0.0
  Entropy= -0.0

Test 2: ['cat', 'dog', 'cat', 'dog']
  Gini   = 0.5
  Entropy= 1.0

Test 3: ['cat', 'dog', 'cat', 'cat', 'dog']
  Gini   = 0.48
  Entropy= 0.971

Test 4: []
  Gini   = 0.0
  Entropy= 0.0

--- counting trace ---
  after seeing 'a': {'a': 1}
  after seeing 'b': {'a': 1, 'b': 1}
  after seeing 'a': {'a': 2, 'b': 1}
  after seeing 'c': {'a': 2, 'b': 1, 'c': 1}
  after seeing 'a': {'a': 3, 'b': 1, 'c': 1}
  after seeing 'b': {'a': 3, 'b': 2, 'c': 1}


Find the best split

In [6]:
def information_gain(parent_labels,left_labels,right_labels,criterion="gini"):
    measure=gini_impurity if criterion=="gini" else entropy
    n=len(parent_labels)
    n_left=len(left_labels)
    n_right=len(right_labels)
    if n_left==0 or n_right==0:
        return 0.0
    parent_impurity=measure(parent_labels)
    child_impurity = (
        (n_left / n) * measure(left_labels) +
        (n_right / n) * measure(right_labels)
    )
    return parent_impurity-child_impurity


In [7]:
# ---- Test 1: PERFECT split (Gini) ----
# Parent: 2 cats + 2 dogs (mixed). Split puts all cats left, all dogs right.
parent = ['cat', 'cat', 'dog', 'dog']
left   = ['cat', 'cat']
right  = ['dog', 'dog']
print("Test 1 (perfect split, gini):", information_gain(parent, left, right, "gini"))
# parent Gini = 0.5, both children pure (0). IG = 0.5 - 0 = 0.5


# ---- Test 1b: PERFECT split (entropy) ----
print("Test 1b (perfect split, entropy):", information_gain(parent, left, right, "entropy"))
# parent H = 1.0, both children 0. IG = 1.0 - 0 = 1.0


# ---- Test 2: NO-OP split (all children still mixed) ----
# Parent same, but split puts a cat+dog on each side → no reduction.
left2  = ['cat', 'dog']
right2 = ['cat', 'dog']
print("Test 2 (useless split, gini):", information_gain(parent, left2, right2, "gini"))
# parent Gini = 0.5, children Gini = 0.5 each, weighted = 0.5. IG = 0.0


# ---- Test 3: DEGENERATE split (one side empty) ----
left3  = ['cat', 'cat', 'dog', 'dog']
right3 = []
print("Test 3 (empty child):", information_gain(parent, left3, right3, "gini"))
# returns 0.0 by the guard clause


# ---- Test 4: UNBALANCED but still useful split ----
# Parent: 3 cats + 1 dog. Left gets 2 cats + 1 dog, right gets 1 cat.
parent4 = ['cat', 'cat', 'cat', 'dog']
left4   = ['cat', 'cat', 'dog']
right4  = ['cat']
print("Test 4 (unbalanced, gini):", round(information_gain(parent4, left4, right4, "gini"), 4))
# parent Gini   = 1 - (3/4)^2 - (1/4)^2 = 0.375
# left Gini     = 1 - (2/3)^2 - (1/3)^2 = 0.4444
# right Gini    = 0
# weighted child = (3/4)*0.4444 + (1/4)*0 = 0.3333
# IG = 0.375 - 0.3333 = 0.0417


# ---- Test 5: UNBALANCED split (entropy) ----
print("Test 5 (unbalanced, entropy):", round(information_gain(parent4, left4, right4, "entropy"), 4))
# parent H       = -(0.75*log2 0.75 + 0.25*log2 0.25) = 0.8113
# left H         = -(2/3*log2 2/3 + 1/3*log2 1/3)     = 0.9183
# right H        = 0
# weighted child = (3/4)*0.9183 = 0.6887
# IG = 0.8113 - 0.6887 = 0.1226


# ---- Test 6: NO GAIN at all (split a pure parent) ----
parent6 = ['cat', 'cat', 'cat']
left6   = ['cat', 'cat']
right6  = ['cat']
print("Test 6 (pure parent, gini):", information_gain(parent6, left6, right6, "gini"))
# parent Gini = 0, both children 0, IG = 0.0


# ---- Test 7: NO GAIN at all (pure parent, entropy) ----
print("Test 7 (pure parent, entropy):", information_gain(parent6, left6, right6, "entropy"))
# parent H = 0, IG = 0.0

Test 1 (perfect split, gini): 0.5
Test 1b (perfect split, entropy): 1.0
Test 2 (useless split, gini): 0.0
Test 3 (empty child): 0.0
Test 4 (unbalanced, gini): 0.0417
Test 5 (unbalanced, entropy): 0.1226
Test 6 (pure parent, gini): 0.0
Test 7 (pure parent, entropy): 0.0
